# Kiva Lending Patterns - Data Cleaning

**Dataset:** Historical Kiva loan records  
**Separator:** `#`  

Loads the raw dataset, inspects it, applies cleaning steps, and saves the cleaned CSV.

In [ ]:
import pandas as pd
import numpy as np
import os

### 1. Loading the Dataset

The file uses `#` as the column separator. The first column in the file is an unnamed row index, so `index_col=0` is used to make it the DataFrame index. `skipinitialspace=True` strips the leading space that appears after each `#` in the header, giving clean column names without further post-processing.

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'data_abschlussprojekt.csv')

df = pd.read_csv(
    DATA_PATH,
    sep='#',
    index_col=0,
    skipinitialspace=True
)

print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

### 2. Initial Inspection

Before any cleaning, inspect the raw state of the data: column names, data types, first rows, summary statistics, missing values, and duplicates.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
df.describe(include='all')

#### Missing Values

Count and percentage of missing values per column.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})
print("Columns with missing values:")
print(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))

#### Duplicate Rows

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes:,}")
print(f"Unique rows: {len(df) - n_dupes:,} of {len(df):,} ({(1 - n_dupes/len(df))*100:.2f}%)")

#### Categorical Column Values

Check unique values in key categorical columns to catch encoding inconsistencies before cleaning.

In [ ]:
print("Unique sectors:", sorted(df['sector'].dropna().unique()))
print("\nUnique repayment intervals:", sorted(df['repayment_interval'].dropna().unique()))

#### Numerical Column Ranges

Check for negative or zero values in columns that should be strictly positive.

In [ ]:
for col in ['funded_amount', 'loan_amount', 'term_in_months', 'lender_count']:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, zeros={zero:,}, negatives={neg:,}")

### 3. Data Cleaning

The following cleaning steps are applied in sequence. Each step documents the decision and its justification.

#### Step 1 - Strip Whitespace from String Columns

String values in the raw file may carry leading or trailing whitespace. This is stripped uniformly across all object columns without losing any data.

In [ ]:
str_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"Stripped whitespace from {len(str_cols)} string columns: {str_cols}")

#### Step 2 - Duplicate Rows

Exact duplicate rows are dropped only if confirmed. A row is an exact duplicate if every column value matches another row. Kiva loan records are not expected to repeat; genuine duplicates are data entry errors.

In [ ]:
n_before = len(df)
df = df.drop_duplicates()
n_dropped = n_before - len(df)
print(f"Rows before: {n_before:,}")
print(f"Exact duplicates dropped: {n_dropped:,}")
print(f"Rows after: {len(df):,}")

#### Step 3 - Missing Values: `region`

`region` is a free-text field describing the sub-national area within a country. Kiva data commonly has a large share of missing regions because lenders are not required to supply them. This step records the count; the fill to `'Not specified'` happens in Step 9 together with all remaining string columns.

In [ ]:
missing_region = df['region'].isnull().sum()
pct_region = missing_region / len(df) * 100
print(f"Missing region: {missing_region:,} ({pct_region:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 4 - Missing Values: `borrower_genders`

`borrower_genders` holds one or more gender labels separated by commas. Missing values indicate the information was not supplied. This step records the count; the fill happens in Step 9.

In [ ]:
missing_bg = df['borrower_genders'].isnull().sum()
pct_bg = missing_bg / len(df) * 100
print(f"Missing borrower_genders: {missing_bg:,} ({pct_bg:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 5 - Missing Values: `use`

`use` is a free-text description of how the borrower intends to use the loan. This step records the count; the fill happens in Step 9.

In [ ]:
missing_use = df['use'].isnull().sum()
pct_use = missing_use / len(df) * 100
print(f"Missing use: {missing_use:,} ({pct_use:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 6 - Missing Values: `country_code`

Eight rows have a missing `country_code` but a populated `country` value of 'Namibia'. The ISO 3166-1 alpha-2 code for Namibia is 'NA'. These eight rows are filled rather than dropped because the country is unambiguously known.

In [ ]:
missing_cc = df['country_code'].isnull().sum()
print(f"Missing country_code before fill: {missing_cc:,}")
print("Countries for these rows:")
print(df.loc[df['country_code'].isnull(), 'country'].value_counts().to_string())

df.loc[df['country_code'].isnull() & (df['country'] == 'Namibia'), 'country_code'] = 'NA'

remaining_cc = df['country_code'].isnull().sum()
print(f"\nMissing country_code after fill: {remaining_cc:,}")

#### Step 7 - Missing Values: `term_in_months`

Confirm whether `term_in_months` has any missing values. If present, they would be retained rather than imputed, since imputing duration without supporting data would distort the distribution.

In [ ]:
missing_term = df['term_in_months'].isnull().sum()
pct_term = missing_term / len(df) * 100
print(f"Missing term_in_months: {missing_term:,} ({pct_term:.1f}% of rows)")

#### Step 8 - Remaining Missing Values Check

Confirm which columns still have NaN before the fill step.

In [ ]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if remaining_missing.empty:
    print("No NaN remaining before fill step.")
else:
    print("Columns with NaN before fill:")
    for col, cnt in remaining_missing.items():
        print(f"  {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

#### Step 9 - Fill Remaining String NaN with 'Not specified'

All remaining NaN values in string columns are filled with `'Not specified'`. This makes the absence of data explicit and visible in groupby operations and visualizations instead of being silently excluded. The label `'Not specified'` is chosen over `'Unknown'` to avoid implying the information exists but is unclear -- it simply was not recorded.

In [ ]:
str_cols_with_nan = df.select_dtypes(include=['object', 'str']).columns[
    df.select_dtypes(include=['object', 'str']).isnull().any()
].tolist()

for col in str_cols_with_nan:
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna('Not specified')
    print(f"  {col}: filled {filled:,} NaN with 'Not specified'")

print(f"\nTotal NaN remaining in string columns: {df.select_dtypes(include=['object','str']).isnull().sum().sum()}")

#### Step 10 - Numerical Validity

Records with a `loan_amount` of zero or below are not meaningful loan records. `funded_amount` can legitimately be zero (partially funded) but not negative. Rows failing these checks are dropped with a count reported.

In [ ]:
n_before = len(df)

invalid_loan = df['loan_amount'] <= 0
invalid_funded = df['funded_amount'] < 0
invalid_mask = invalid_loan | invalid_funded

print(f"Rows with loan_amount <= 0:  {invalid_loan.sum():,}")
print(f"Rows with funded_amount < 0: {invalid_funded.sum():,}")
print(f"Total invalid rows to drop:  {invalid_mask.sum():,}")

df = df[~invalid_mask].copy()
print(f"\nRows after numerical validity filter: {len(df):,} (dropped {n_before - len(df):,})")

#### Step 11 - Unusual `term_in_months` Values

Inspect the distribution of `term_in_months` to identify unusually short or long durations. Unusual values are noted but not removed unless they are clearly data entry errors (negative or zero), since genuine long-term loans exist.

In [ ]:
print("term_in_months distribution:")
print(df['term_in_months'].describe())
print(f"\nRows with term_in_months <= 0: {(df['term_in_months'] <= 0).sum():,}")
print(f"Rows with term_in_months > 120 (more than 10 years): {(df['term_in_months'] > 120).sum():,}")

Drop rows where `term_in_months` is zero or negative; these cannot represent valid loan durations.

In [ ]:
n_before = len(df)
invalid_term = df['term_in_months'].notna() & (df['term_in_months'] <= 0)
df = df[~invalid_term].copy()
print(f"Dropped {n_before - len(df):,} rows with term_in_months <= 0.")
print(f"Rows remaining: {len(df):,}")

#### Step 12 - Derived Feature: `funding_gap`

`funding_gap` is the difference between the requested amount and the funded amount. A positive value means the loan was partially funded. A negative value means more was disbursed than originally requested; the assignment context confirms this can happen and these records are kept. Zero means fully funded.

In [ ]:
df['funding_gap'] = df['loan_amount'] - df['funded_amount']

fully_funded = (df['funding_gap'] == 0).sum()
partially_funded = (df['funding_gap'] > 0).sum()
over_funded = (df['funding_gap'] < 0).sum()

print(f"Fully funded (gap = 0):          {fully_funded:,} ({fully_funded/len(df)*100:.1f}%)")
print(f"Partially funded (gap > 0):      {partially_funded:,} ({partially_funded/len(df)*100:.1f}%)")
print(f"Over-funded (gap < 0):           {over_funded:,} ({over_funded/len(df)*100:.1f}%)")

print("\nfunding_gap statistics:")
print(df['funding_gap'].describe())

#### Step 13 - Final Cleaned Dataset Summary

In [ ]:
print("Final cleaned dataset:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print("\nRemaining missing values:")
final_missing = df.isnull().sum()
print(final_missing[final_missing > 0])
print("\nFirst rows:")
df.head()

### 4. Save Cleaned Dataset

The cleaned and processed dataset is saved as a CSV to the `data` directory for use in subsequent analysis and visualization steps.

In [ ]:
OUTPUT_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
df.to_csv(OUTPUT_PATH)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")